In [0]:
%sql
SELECT * FROM samples.tpch.orders LIMIT 5;
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

SHOW SCHEMAS IN workspace;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files;


In [0]:
%sql
LIST '/Volumes/workspace/bronze/raw_files/';

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.orders AS
SELECT *, current_timestamp() AS _ingested_at
FROM samples.tpch.orders;

In [0]:
%python
from pyspark.sql import functions as F

df = (spark.read.format("json")
      .load("/Volumes/workspace/bronze/raw_files/web_events.jsonl")
      .withColumn("_ingested_at", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path")))

df.write.mode("overwrite").saveAsTable("workspace.bronze.web_events")

In [0]:
%sql
SELECT * FROM workspace.bronze.web_events;